In [0]:
BRONZE = "/Volumes/workspace/default/bronze"
SILVER = "/Volumes/workspace/default/silver"

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.silver")
print(f"Silver path: {SILVER}")

In [0]:
from pyspark.sql.functions import col, trim, lower, when, lit, desc, row_number, current_timestamp
from pyspark.sql.types import IntegerType, BooleanType, DateType, DecimalType
from pyspark.sql.window import Window

df = spark.read.parquet(f"{BRONZE}/employee")

# snake_case rename
df = (df
    .withColumnRenamed("EmployeeId",    "employee_id")
    .withColumnRenamed("FirstName",     "first_name")
    .withColumnRenamed("LastName",      "last_name")
    .withColumnRenamed("Email",         "email")
    .withColumnRenamed("DepartmentId",  "department_id")
    .withColumnRenamed("ManagerId",     "manager_id")
    .withColumnRenamed("JobTitle",      "job_title")
    .withColumnRenamed("JoinDate",      "join_date")
    .withColumnRenamed("IsActive",      "is_active")
    .withColumnRenamed("AnnualLeaveDays", "annual_leave_days")
)

# Cast + validate
df = df.withColumn("employee_id",   col("employee_id").cast(IntegerType())) \
       .withColumn("department_id", col("department_id").cast(IntegerType())) \
       .withColumn("join_date",     col("join_date").cast(DateType())) \
       .withColumn("is_active",     col("is_active").cast(BooleanType()))

df = df.withColumn("_is_valid",
    when(col("employee_id").isNull(), lit(False))
    .when(col("first_name").isNull() | (trim(col("first_name")) == ""), lit(False))
    .otherwise(lit(True))
).withColumn("_validation_error",
    when(col("employee_id").isNull(), lit("employee_id is null"))
    .when(col("first_name").isNull() | (trim(col("first_name")) == ""), lit("first_name is null/empty"))
    .otherwise(lit(None).cast("string"))
)

# Dedup: keep latest extraction per employee_id
w = Window.partitionBy("employee_id").orderBy(desc("_extracted_at"))
df = df.withColumn("_rn", row_number().over(w)).filter("_rn = 1").drop("_rn")

df.write.mode("overwrite").parquet(f"{SILVER}/employee")
print(f"âœ“ employee: {df.count()} rows ({df.filter('_is_valid = false').count()} invalid)")


In [0]:
df = spark.read.parquet(f"{BRONZE}/department")

df = (df
    .withColumnRenamed("DepartmentId",   "department_id")
    .withColumnRenamed("DepartmentName", "department_name")
    .withColumnRenamed("ManagerId",      "manager_id")
)

df = df.withColumn("department_id", col("department_id").cast(IntegerType()))

df = df.withColumn("_is_valid",
    when(col("department_id").isNull(), lit(False)).otherwise(lit(True))
).withColumn("_validation_error",
    when(col("department_id").isNull(), lit("department_id is null")).otherwise(lit(None).cast("string"))
)

w = Window.partitionBy("department_id").orderBy(desc("_extracted_at"))
df = df.withColumn("_rn", row_number().over(w)).filter("_rn = 1").drop("_rn")

df.write.mode("overwrite").parquet(f"{SILVER}/department")
print(f"âœ“ department: {df.count()} rows ({df.filter('_is_valid = false').count()} invalid)")


In [0]:
df = spark.read.parquet(f"{BRONZE}/leavetype")

df = (df
    .withColumnRenamed("LeaveTypeId",             "leave_type_id")
    .withColumnRenamed("LeaveTypeName",           "leave_type_name")
    .withColumnRenamed("DefaultDaysPerYear",      "default_days_per_year")
    .withColumnRenamed("IsCarryForwardAllowed",   "is_carry_forward_allowed")
    .withColumnRenamed("MaxCarryForwardDays",     "max_carry_forward_days")
)

df = df.withColumn("leave_type_id",           col("leave_type_id").cast(IntegerType())) \
       .withColumn("default_days_per_year",   col("default_days_per_year").cast(IntegerType())) \
       .withColumn("is_carry_forward_allowed",col("is_carry_forward_allowed").cast(BooleanType()))

df = df.withColumn("_is_valid",
    when(col("leave_type_id").isNull(), lit(False)).otherwise(lit(True))
).withColumn("_validation_error",
    when(col("leave_type_id").isNull(), lit("leave_type_id is null")).otherwise(lit(None).cast("string"))
)

w = Window.partitionBy("leave_type_id").orderBy(desc("_extracted_at"))
df = df.withColumn("_rn", row_number().over(w)).filter("_rn = 1").drop("_rn")

df.write.mode("overwrite").parquet(f"{SILVER}/leavetype")
print(f"âœ“ leavetype: {df.count()} rows ({df.filter('_is_valid = false').count()} invalid)")


In [0]:
df = spark.read.parquet(f"{BRONZE}/leaverequest")

df = (df
    .withColumnRenamed("LeaveRequestId", "leave_request_id")
    .withColumnRenamed("EmployeeId",     "employee_id")
    .withColumnRenamed("LeaveTypeId",    "leave_type_id")
    .withColumnRenamed("StartDate",      "start_date")
    .withColumnRenamed("EndDate",        "end_date")
    .withColumnRenamed("TotalDays",      "total_days")
    .withColumnRenamed("Status",         "status")
    .withColumnRenamed("Reason",         "reason")
    .withColumnRenamed("ApprovedBy",     "approved_by")
    .withColumnRenamed("ApprovedAt",     "approved_at")
    .withColumnRenamed("CreatedAt",      "created_at")
    .withColumnRenamed("UpdatedAt",      "updated_at")
)

df = df.withColumn("leave_request_id", col("leave_request_id").cast(IntegerType())) \
       .withColumn("employee_id",      col("employee_id").cast(IntegerType())) \
       .withColumn("leave_type_id",    col("leave_type_id").cast(IntegerType())) \
       .withColumn("total_days",       col("total_days").cast(DecimalType(5, 1))) \
       .withColumn("start_date",       col("start_date").cast(DateType())) \
       .withColumn("end_date",         col("end_date").cast(DateType()))

df = df.withColumn("_is_valid",
    when(col("leave_request_id").isNull(), lit(False))
    .when(col("employee_id").isNull(), lit(False))
    .when(col("start_date") > col("end_date"), lit(False))
    .otherwise(lit(True))
).withColumn("_validation_error",
    when(col("leave_request_id").isNull(), lit("leave_request_id is null"))
    .when(col("employee_id").isNull(), lit("employee_id is null"))
    .when(col("start_date") > col("end_date"), lit("start_date > end_date"))
    .otherwise(lit(None).cast("string"))
)

w = Window.partitionBy("leave_request_id").orderBy(desc("_extracted_at"))
df = df.withColumn("_rn", row_number().over(w)).filter("_rn = 1").drop("_rn")

df.write.mode("overwrite").parquet(f"{SILVER}/leaverequest")
print(f"âœ“ leaverequest: {df.count()} rows ({df.filter('_is_valid = false').count()} invalid)")


In [0]:
df = spark.read.parquet(f"{BRONZE}/leavebalance")

df = (df
    .withColumnRenamed("LeaveBalanceId", "leave_balance_id")
    .withColumnRenamed("EmployeeId",     "employee_id")
    .withColumnRenamed("LeaveTypeId",    "leave_type_id")
    .withColumnRenamed("Year",           "year")
    .withColumnRenamed("TotalEntitled",  "total_entitled")
    .withColumnRenamed("TotalUsed",      "total_used")
    .withColumnRenamed("CarryForward",   "carry_forward")
    .withColumnRenamed("Remaining",      "remaining")
    .withColumnRenamed("CreatedAt",      "created_at")
    .withColumnRenamed("UpdatedAt",      "updated_at")
)

df = df.withColumn("leave_balance_id", col("leave_balance_id").cast(IntegerType())) \
       .withColumn("employee_id",      col("employee_id").cast(IntegerType())) \
       .withColumn("leave_type_id",    col("leave_type_id").cast(IntegerType())) \
       .withColumn("total_entitled",   col("total_entitled").cast(DecimalType(5,1))) \
       .withColumn("total_used",       col("total_used").cast(DecimalType(5,1))) \
       .withColumn("remaining",        col("remaining").cast(DecimalType(6,1)))

df = df.withColumn("_is_valid",
    when(col("leave_balance_id").isNull(), lit(False)).otherwise(lit(True))
).withColumn("_validation_error",
    when(col("leave_balance_id").isNull(), lit("leave_balance_id is null")).otherwise(lit(None).cast("string"))
)

w = Window.partitionBy("leave_balance_id").orderBy(desc("_extracted_at"))
df = df.withColumn("_rn", row_number().over(w)).filter("_rn = 1").drop("_rn")

df.write.mode("overwrite").parquet(f"{SILVER}/leavebalance")
print(f"âœ“ leavebalance: {df.count()} rows ({df.filter('_is_valid = false').count()} invalid)")


In [0]:
df = spark.read.parquet(f"{BRONZE}/attendance")

# Already snake_case from generation â€” just cast
df = df.withColumn("employee_id", col("EmployeeId").cast(IntegerType())) \
       .withColumn("work_date",   col("WorkDate").cast(DateType())) \
       .withColumn("check_in_time",  col("CheckInTime")) \
       .withColumn("check_out_time", col("CheckOutTime")) \
       .withColumn("is_present",     col("IsPresent").cast(BooleanType())) \
       .drop("EmployeeId", "WorkDate", "CheckInTime", "CheckOutTime", "IsPresent")

df = df.withColumn("_is_valid",
    when(col("employee_id").isNull(), lit(False))
    .when(col("work_date").isNull(), lit(False))
    .otherwise(lit(True))
).withColumn("_validation_error",
    when(col("employee_id").isNull(), lit("employee_id is null"))
    .when(col("work_date").isNull(), lit("work_date is null"))
    .otherwise(lit(None).cast("string"))
)

w = Window.partitionBy("employee_id", "work_date").orderBy(desc("_extracted_at"))
df = df.withColumn("_rn", row_number().over(w)).filter("_rn = 1").drop("_rn")

df.write.mode("overwrite").parquet(f"{SILVER}/attendance")
print(f"âœ“ attendance: {df.count()} rows ({df.filter('_is_valid = false').count()} invalid)")


In [0]:
tables = ["employee", "department", "leavetype", "leaverequest", "leavebalance", "attendance"]
for t in tables:
    df = spark.read.parquet(f"{SILVER}/{t}")
    invalid = df.filter("_is_valid = false").count()
    print(f"{SILVER}/{t}: {df.count()} rows, {invalid} invalid")
